In [3]:
"""
render_trajectory_video.py

Step 5 (visualization) of the glass pipeline: render a video showing
the time evolution of tracked particle trajectories -- a clean,
schematic view (colored circles on a black background) rather than
the raw grainy video, useful as a sanity check before MSD analysis.

Rendering choices (as discussed):
  - black background
  - particles colored by class: bright ocean blue (small) vs
    green-yellow (big) -- adjust COLOR_SMALL / COLOR_BIG below if
    the assignment should be swapped
  - circle radius = that particle's time-averaged fitted size (px),
    scaled by RADIUS_SCALE for visibility (fitted sizes are only a
    few px, which would be nearly invisible at 1:1)
  - fading trail: each video frame, the whole canvas is dimmed by
    TRAIL_DECAY before drawing new positions on top, so recent
    positions stay bright and older ones fade smoothly to black.
    This is a standard, efficient way to render trails without
    tracking per-particle path history explicitly.
  - 30 fps playback (4000 data-frames -> ~2.2 minutes of video),
    NOT the real 1 fps acquisition rate.

CANVAS SIZE: auto-computed from the actual range of x/y positions in
the data (plus a margin), NOT necessarily identical to the original
crop dimensions. If you need it to exactly match the crop (e.g. to
overlay on the raw video later), set CANVAS_WIDTH/CANVAS_HEIGHT
manually instead of leaving them as None.

Input:  linked_trajectories_classified.csv from classify_by_size.py
Output: an .mp4 video of the animated trajectories
"""

import os
import subprocess
import numpy as np
import pandas as pd
import cv2

# ----------------------------------------------------------------------
# 1. CONFIG -- edit these values
# ----------------------------------------------------------------------

INPUT_CSV = "/Volumes/Expansion/recordings/linked_trajectories_classified.csv"   # <-- fill in: path to linked_trajectories_classified.csv
OUTPUT_DIR = os.path.dirname(INPUT_CSV)  # <-- fill in: folder to save the video into
OUTPUT_FILENAME = "trajectory_evolution.mp4"

CANVAS_WIDTH = None   # None -> auto-compute from data extent + margin
CANVAS_HEIGHT = None  # None -> auto-compute from data extent + margin
CANVAS_MARGIN_PX = 20  # only used when auto-computing

FPS = 30
TRAIL_DECAY = 0.85  # fraction of brightness kept each frame (0-1); lower = shorter trail

RADIUS_SCALE = 3.0  # multiply fitted size (px) by this for visibility

PRINT_EVERY = 100  # print a progress update every this many rendered frames

# colors as (R, G, B) -- converted to BGR internally for OpenCV
COLOR_SMALL_RGB = (0, 180, 255)    # bright ocean blue
COLOR_BIG_RGB = (173, 255, 47)     # green-yellow


def main():
    if INPUT_CSV is None or OUTPUT_DIR is None:
        raise ValueError("Set INPUT_CSV and OUTPUT_DIR before running.")

    df = pd.read_csv(INPUT_CSV)
    for col in ("x", "y", "frame", "particle", "class", "size"):
        if col not in df.columns:
            raise ValueError(f"Expected column '{col}' not found in input CSV.")

    # ------------------------------------------------------------------
    # 2. Determine canvas size
    # ------------------------------------------------------------------

    if CANVAS_WIDTH is None or CANVAS_HEIGHT is None:
        x_min, x_max = df["x"].min(), df["x"].max()
        y_min, y_max = df["y"].min(), df["y"].max()
        width = int(np.ceil(x_max - x_min)) + 2 * CANVAS_MARGIN_PX
        height = int(np.ceil(y_max - y_min)) + 2 * CANVAS_MARGIN_PX
        x_offset = x_min - CANVAS_MARGIN_PX
        y_offset = y_min - CANVAS_MARGIN_PX
        print(f"Auto-computed canvas: {width}x{height} px "
              f"(from data extent x[{x_min:.1f},{x_max:.1f}] "
              f"y[{y_min:.1f},{y_max:.1f}], margin {CANVAS_MARGIN_PX}px)")
    else:
        width, height = CANVAS_WIDTH, CANVAS_HEIGHT
        x_offset, y_offset = 0, 0
        print(f"Using fixed canvas: {width}x{height} px")

    # ------------------------------------------------------------------
    # 3. Per-particle average size (for radius) and class (for color)
    #    -- same idea as classify_by_size.py, computed fresh here so
    #    this script only depends on the CSV, not on re-running that step
    # ------------------------------------------------------------------

    avg_size_per_particle = df.groupby("particle")["size"].mean()
    class_per_particle = df.groupby("particle")["class"].first()

    color_small_bgr = COLOR_SMALL_RGB[::-1]
    color_big_bgr = COLOR_BIG_RGB[::-1]

    # ------------------------------------------------------------------
    # 4. Set up an ffmpeg subprocess as the video writer.
    #
    #    We deliberately do NOT use cv2.VideoWriter here. OpenCV's video
    #    writing on macOS is unreliable -- it can report success
    #    (writer.isOpened() == True) and produce a large, seemingly valid
    #    file, while actually writing a malformed H.264/MPEG-4 stream
    #    that neither QuickTime nor even permissive players like IINA can
    #    decode. This happened in practice with this exact script.
    #
    #    Instead, we pipe raw frames directly into ffmpeg via stdin and
    #    let ffmpeg -- a far more standard, reliable encoder -- do the
    #    actual H.264 encoding. This guarantees a widely-compatible file.
    # ------------------------------------------------------------------

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

    ffmpeg_cmd = [
        "ffmpeg",
        "-y",  # overwrite output file if it already exists
        "-f", "rawvideo",
        "-vcodec", "rawvideo",
        "-pix_fmt", "bgr24",          # matches the BGR frames we hand it (OpenCV convention)
        "-s", f"{width}x{height}",
        "-r", str(FPS),
        "-i", "-",                    # read raw frames from stdin
        "-an",                        # no audio
        "-vcodec", "libx264",         # standard, widely-compatible H.264 encoder
        "-pix_fmt", "yuv420p",        # required by QuickTime/most players
        output_path,
    ]

    ffmpeg_process = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE)

    # ------------------------------------------------------------------
    # 5. Render frame by frame
    #    Canvas is kept as float32 so repeated dimming (TRAIL_DECAY)
    #    doesn't lose precision the way repeated uint8 multiplication
    #    would (rounding to 0 too fast, killing the fade effect).
    # ------------------------------------------------------------------

    canvas = np.zeros((height, width, 3), dtype=np.float32)

    frames_sorted = sorted(df["frame"].unique())
    print(f"Rendering {len(frames_sorted)} frames at {FPS} fps "
          f"(~{len(frames_sorted)/FPS:.1f}s of video)...")

    grouped_by_frame = df.groupby("frame")

    for i, frame_num in enumerate(frames_sorted):
        canvas *= TRAIL_DECAY

        frame_rows = grouped_by_frame.get_group(frame_num)
        for _, row in frame_rows.iterrows():
            particle_id = row["particle"]
            x = row["x"] - x_offset
            y = row["y"] - y_offset
            radius = max(1, int(round(avg_size_per_particle[particle_id] * RADIUS_SCALE)))
            color = color_small_bgr if row["class"] == "small" else color_big_bgr

            cv2.circle(canvas, (int(round(x)), int(round(y))), radius,
                       color, thickness=-1, lineType=cv2.LINE_AA)

        frame_uint8 = np.clip(canvas, 0, 255).astype(np.uint8)
        ffmpeg_process.stdin.write(frame_uint8.tobytes())

        if (i + 1) % PRINT_EVERY == 0 or (i + 1) == len(frames_sorted):
            print(f"  rendered {i+1}/{len(frames_sorted)} frames")

    ffmpeg_process.stdin.close()
    ffmpeg_process.wait()
    if ffmpeg_process.returncode != 0:
        raise RuntimeError(
            f"ffmpeg exited with error code {ffmpeg_process.returncode} -- "
            "check the ffmpeg output above for details."
        )
    print(f"\nDone. Saved trajectory video to:\n  {output_path}")


if __name__ == "__main__":
    main()

KeyboardInterrupt: 